# Prepare the declared 3PTB case and admit its execution plan

This installed-package walkthrough makes the existing scientific prescription
visible through public MolSysMT and BindingMT APIs. **Run All prepares one
receptor and one rigid ligand and validates two planned problems; it launches
no maps, energy evaluation, docking, optimization, RMSD or viewer action.**

The final Markdown describes a separately registered execution follow-up. It is
not an executed result or an automatic continuation of this notebook.

Inputs are the original public 3PTB PDB and the retained admission-declaration
JSON, each with a caller-supplied external SHA-256, plus a new output directory.
Set `BINDINGMT_3PTB_SOURCE`, `BINDINGMT_3PTB_SOURCE_SHA256`,
`BINDINGMT_3PTB_DECLARATIONS`, `BINDINGMT_3PTB_DECLARATIONS_SHA256` and
`BINDINGMT_3PTB_OUTPUT` before starting the kernel, or edit those assignments.
No repository import, `devtools`, private archive path or SDK is needed.

The declaration file supplies explicit state choices and caller-declared atom
maps. Its stored molecular snapshots are not restored or interpreted. This is a
case-specific scientific prescription, not an automatic preparation default,
atom matcher, generic archive or claim of experimental protonation. Molecular
operations and all format handling remain in MolSysMT.

In [ ]:
import gzip
import hashlib
import importlib.metadata as metadata
import json
import os
import platform
import sys
from pathlib import Path

import molsysmt as msm
import numpy as np
import pyunitwizard as puw
from smonitor.integrations import CatalogException

import bindingmt as dmt

source_path = Path(os.environ['BINDINGMT_3PTB_SOURCE']).expanduser().resolve()
source_sha256 = os.environ['BINDINGMT_3PTB_SOURCE_SHA256'].lower()
declaration_path = (
    Path(os.environ['BINDINGMT_3PTB_DECLARATIONS']).expanduser().resolve()
)
declaration_sha256 = os.environ['BINDINGMT_3PTB_DECLARATIONS_SHA256'].lower()
output = Path(os.environ['BINDINGMT_3PTB_OUTPUT']).expanduser().resolve()
prefix = Path(sys.prefix).resolve()
for module in (dmt, msm, puw):
    assert Path(module.__file__).resolve().is_relative_to(prefix), (
        'Use an installed kernel without source overlays.'
    )
distribution = metadata.distribution('bindingmt')
installation = json.loads(distribution.read_text('direct_url.json') or '{}')
assert not installation.get('dir_info', {}).get('editable', False)
assert dmt.__version__ == metadata.version('bindingmt')
versions = {
    name: metadata.version(name)
    for name in ('bindingmt', 'molsysmt', 'pyunitwizard', 'rdkit')
}
print({'python': platform.python_version(), 'installed_versions': versions})

## Bind the source and the declared scientific choices

Expected file digests must come from a trusted source. The admitted choice record
is read as JSON; only its explicit decisions/maps and original prepared-value
controls are used. It does not become a native-system persistence format.
MolSysMT handles the compressed molecular source and its molecular conversion.
The output directory must not exist, preventing accidental overwrite/replay.

In [ ]:
for path, expected in (
    (source_path, source_sha256),
    (declaration_path, declaration_sha256),
):
    if len(expected) != 64 or any(char not in '0123456789abcdef' for char in expected):
        raise ValueError('Supply externally expected SHA-256 identities.')
    if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        raise ValueError('Input bytes differ from their externally expected identity.')
raw_declarations = declaration_path.read_bytes()
if declaration_path.suffix == '.gz':
    raw_declarations = gzip.decompress(raw_declarations)
declarations = json.loads(raw_declarations)
assert declarations['schema'] == 'bindingmt.3ptb_molecular_admission@1'
assert declarations['compressed_source_sha256'] == source_sha256
plain_source = msm.convert(source_path, to_form='string:pdb_text')
assert (
    hashlib.sha256(plain_source.encode()).hexdigest() == declarations['source_sha256']
)
source = msm.convert(source_path, to_form='molsysmt.MolSys', get_missing_bonds=False)
counts = msm.get(
    source,
    n_atoms=True,
    n_groups=True,
    n_structures=True,
    n_bonds=True,
    output_type='dictionary',
)
assert counts == {'n_atoms': 1701, 'n_groups': 287, 'n_structures': 1, 'n_bonds': 21}
alternate_locations = msm.get(source, element='atom', alternate_location=True)
assert all(
    value in ('', ' ', None) for value in np.asarray(alternate_locations).reshape(-1)
)
source_ids = list(msm.get(source, element='atom', atom_id=True))
source_coordinates = puw.get_value(
    msm.get(source, coordinates=True), to_unit='angstrom'
).copy()
output.mkdir(parents=True, exist_ok=False)
receptor = msm.extract(
    source, selection="molecule_type=='protein'", structure_indices=0
)
reference = msm.extract(source, selection="group_name=='BEN'", structure_indices=0)
assert (
    msm.get(receptor, n_atoms=True) == 1629 and msm.get(receptor, n_groups=True) == 223
)
assert msm.get(reference, n_atoms=True) == 9
assert set(msm.get(receptor, element='atom', chain_id=True)) == {'A'}
assert set(msm.get(reference, element='atom', chain_id=True)) == {'A'}
print(
    {'source_counts': counts, 'receptor_heavy_atoms': 1629, 'reference_heavy_atoms': 9}
)

## State the hypothesis, then construct the provider's templates

The receptor is dry chain A: no calcium, waters or crystal neighbours. The
prescription declares HID57, HIE40/HIE91, six deposited CYX links, ammonium and
carboxylate termini. It is a choice, not pH prediction. The ligand is explicit
benzamidinium +1 (`NC(=[NH2+])c1ccccc1`), independently of neutral CCD BEN.

The 1,629/9 template-to-source maps are already-declared inputs. MolSysMT checks
and applies them; this example derives no molecular correspondence locally.
The ligand template checksum below identifies the **SMILES prescription bytes**,
not a canonical native graph. The scope is explicit and no historical native
snapshot digest is rebound to the new template.

In [ ]:
residue_states = declarations['receptor']['declared_group_states']
group_ids = list(msm.get(receptor, element='group', group_id=True))
histidine_choices = {
    str(group_ids[index]): state
    for index, state in enumerate(residue_states)
    if state in ('HID', 'HIE', 'HIP')
}
assert histidine_choices == {'40': 'HIE', '57': 'HID', '91': 'HIE'}
disulfide_pairs = declarations['receptor']['disulfide_selected_group_pairs']
assert declarations['receptor']['disulfide_author_pairs'] == [
    [22, 157],
    [42, 58],
    [128, 232],
    [136, 201],
    [168, 182],
    [191, 220],
]
factory = msm.physchem.get_peptide_chemical_template(
    residue_states,
    n_terminal_state='ammonium',
    c_terminal_state='carboxylate',
    disulfide_group_pairs=disulfide_pairs,
)
receptor_map = np.asarray(
    declarations['receptor']['template_correspondence'], dtype=int
)
assert receptor_map.shape == (1629, 2)
# Verify this supplied identity declaration, without discovering a new match.
for field in ('group_index', 'atom_name'):
    template_values = np.asarray(
        msm.get(factory['template'], element='atom', **{field: True})
    )
    source_values = np.asarray(msm.get(receptor, element='atom', **{field: True}))
    np.testing.assert_array_equal(
        template_values[receptor_map[:, 0]], source_values[receptor_map[:, 1]]
    )
receptor_options = {
    'template': factory['template'],
    'atom_correspondence': receptor_map,
    'template_provenance': factory['template_provenance'],
    'connectivity_policy': 'complete_from_template',
}
smiles = 'smiles:NC(=[NH2+])c1ccccc1'
ligand_template = msm.convert(
    msm.convert(smiles, to_form='rdkit.Mol'), to_form='molsysmt.MolSys'
)
ligand_options = {
    'template': ligand_template,
    'atom_correspondence': declarations['ligand']['template_options'][
        'atom_correspondence'
    ],
    'template_provenance': {
        'identity': 'Declared 3PTB benzamidinium +1 prescription',
        'version': f'MolSysMT {versions["molsysmt"]}; RDKit {versions["rdkit"]}',
        'source_uri': smiles,
        'checksum': 'sha256:' + hashlib.sha256(smiles.encode()).hexdigest(),
        'checksum_scope': 'declared SMILES prescription bytes, not native template bytes',
        'hydrogen_policy': 'stored_counts',
    },
    'connectivity_policy': 'complete_from_template',
}
print(
    {
        'histidines': histidine_choices,
        'disulfide_author_pairs': declarations['receptor']['disulfide_author_pairs'],
        'termini': ['ammonium', 'carboxylate'],
        'ligand': smiles,
        'active_ligand_cuts': [],
    }
)

## Let MolSysMT admit the declarations

An invalid detached map is an informative refusal control, not permission to
repair a source or select another template. Valid compatible reports precede
application. Existing source chemistry is not silently overwritten.

In [ ]:
receptor_assessment = msm.physchem.assess_chemical_template(
    receptor, **receptor_options
)
ligand_assessment = msm.physchem.assess_chemical_template(reference, **ligand_options)
assert receptor_assessment['status'] == ligand_assessment['status'] == 'compatible'
invalid_map = receptor_map.copy()
invalid_map[0, 1] = invalid_map[1, 1]
try:
    msm.physchem.assess_chemical_template(
        receptor, **{**receptor_options, 'atom_correspondence': invalid_map}
    )
except CatalogException as error:
    assert error.extra.get('argument') == 'atom_correspondence'
    map_refusal = {'code': error.code, 'argument': error.extra['argument']}
else:
    raise AssertionError('The invalid caller-declared map was not refused.')
receptor_application = msm.physchem.apply_chemical_template(
    receptor, **receptor_options
)
ligand_application = msm.physchem.apply_chemical_template(reference, **ligand_options)
print(
    {
        'template_assessments': ['compatible', 'compatible'],
        'invalid_map_refused': map_refusal,
    }
)

## Add fixed-state H through MolSysMT

RDKit places H for the declared state, with no pH inference or optimisation.
Atom expansion explicitly permits reported occupancy/B-factor loss. Original
arrays are retained separately below. The case's existing generated-receptor-H
naming rule is declared (`H` plus three uppercase hexadecimal output-index digits)
and applied through `msm.set`; it is presentation, not atom matching or chemistry.
Provider maps bind original versus generated atoms. Heavy identities/coordinates
are checked on those supplied maps without fitting or molecular reconstruction.

In [ ]:
hydrogen_options = {
    'mode': 'fixed_chemical_state',
    'pH': None,
    'engine': 'RDKit',
    'attribute_policy': 'intersection',
}
original_roles = {
    'receptor': receptor_application['molecular_system'],
    'partner': ligand_application['molecular_system'],
}
original_attributes = {}
hydrogen_outputs = {}
for role, molecular_system in original_roles.items():
    original_attributes[role] = {
        'occupancy': msm.get(molecular_system, element='atom', occupancy=True),
        'b_factor': puw.get_value(
            msm.get(molecular_system, element='atom', b_factor=True),
            to_unit='angstrom**2',
        ),
        'b_factor_unit': 'angstrom**2',
    }
    hydrogen_outputs[role] = msm.build.add_missing_hydrogens(
        molecular_system, **hydrogen_options, return_report=True
    )
    expanded = hydrogen_outputs[role]['molecular_system']
    report = hydrogen_outputs[role]['report']
    assert report['original_coordinates_preserved']
    assert sorted(report['dropped_attributes']) == ['b_factor', 'occupancy']
    atom_map = np.asarray(report['atom_correspondence'], dtype=int)
    before = puw.get_value(
        msm.get(molecular_system, coordinates=True), to_unit='angstrom'
    )[0]
    after = puw.get_value(msm.get(expanded, coordinates=True), to_unit='angstrom')[0]
    np.testing.assert_allclose(
        after[atom_map[:, 1]], before[atom_map[:, 0]], rtol=0, atol=1e-12
    )
    before_ids = np.asarray(msm.get(molecular_system, element='atom', atom_id=True))
    after_ids = np.asarray(msm.get(expanded, element='atom', atom_id=True))
    np.testing.assert_array_equal(after_ids[atom_map[:, 1]], before_ids[atom_map[:, 0]])
receptor_h = hydrogen_outputs['receptor']['molecular_system']
ligand_h = hydrogen_outputs['partner']['molecular_system']
generated_h = [
    int(pair[1])
    for pair in hydrogen_outputs['receptor']['report']['parent_hydrogen_pairs']
]
assert generated_h and max(generated_h) < 4096
msm.set(
    receptor_h,
    element='atom',
    selection=generated_h,
    atom_name=[f'H{index:03X}' for index in generated_h],
)
print(
    {
        'H_mode': 'fixed_chemical_state',
        'pH_inference': False,
        'added_H': {
            role: value['report']['n_added_hydrogens']
            for role, value in hydrogen_outputs.items()
        },
        'explicit_attribute_loss': ['b_factor', 'occupancy'],
    }
)

## Choose named models and prepare through BindingMT's provider adapters

Both roles request Gasteiger-Marsili charges and AutoDock4
`chemical_environment@1` types explicitly. MolSysMT performs assignment,
nonpolar-H projection and charge transfer. The ligand stays rigid. Complete named
fields and conserved charge establish this software/profile admission; they do
not certify the chemistry or score model. `unassessed` stays unassessed.

In [ ]:
charge_options = {'method': 'gasteiger_marsili'}
typing_options = {'typing_scheme': 'autodock4', 'method': 'chemical_environment'}
prepared_receptor = dmt.prepare_receptor(
    receptor_h,
    selection='all',
    state_id=declarations['prepared_inputs']['receptor']['prepared']['state_id'],
    charge_options=charge_options,
    typing_options=typing_options,
)
prepared_partner = dmt.prepare_ligand(
    ligand_h,
    selection='all',
    state_id=declarations['prepared_inputs']['rigid']['prepared']['state_id'],
    active_torsion_bonds=[],
    charge_options=charge_options,
    typing_options=typing_options,
)
preparations = {'receptor': prepared_receptor, 'partner': prepared_partner}
charge_audits = {}
preparation_assessments = {}
for role, prepared in preparations.items():
    charge_audits[role] = dmt.audit_preparation_charges(prepared)
    preparation_assessments[role] = dmt.assess_preparation(prepared)
    assert charge_audits[role]['assessment'] == 'consistent'
    assert charge_audits[role]['charge_unit'] == 'elementary_charge'
    assert preparation_assessments[role]['assessment'] == 'unassessed'
    assert preparation_assessments[role]['provisional_reason_codes'] == []
    assert (
        prepared.metadata['atom_type_assignment']['rule_version']
        == 'chemical_environment@1'
    )
assert (
    prepared_receptor.n_atoms,
    prepared_partner.n_atoms,
    prepared_partner.torsion_dof,
) == (2011, 13, 0)
np.testing.assert_allclose(
    charge_audits['receptor']['total_charge'], 6, rtol=0, atol=1e-8
)
np.testing.assert_allclose(
    charge_audits['partner']['total_charge'], 1, rtol=0, atol=1e-8
)
print(
    {
        'prepared_atoms': {'receptor': 2011, 'partner': 13},
        'projected_charge_unit': 'elementary_charge',
        'projected_charges': {
            role: audit['total_charge'] for role, audit in charge_audits.items()
        },
        'preparation_assessment': 'unassessed',
        'ligand_torsions': 0,
    }
)

## Compare only the registered prepared-value controls

These checks compare this reconstruction with the existing prescribed input
values, not with a newly measured experimental state. Historical metadata and
producer identities are not rebound to new reports or writer bytes. The current
native writer can change byte representation while preserving admitted values.

In [ ]:
for role, prepared in preparations.items():
    key = 'receptor' if role == 'receptor' else 'rigid'
    old = declarations['prepared_inputs'][key]['prepared']
    current = prepared.to_dict()
    for field in (
        'state_id',
        'n_atoms',
        'atom_names',
        'group_names',
        'group_ids',
        'atom_types',
    ):
        assert current[field] == old[field], (role, field)
    evidence = declarations['prepared_inputs'][key]
    expected = puw.quantity(evidence['coordinates'], evidence['coordinate_unit'])
    np.testing.assert_allclose(
        puw.get_value(prepared.coordinates, to_unit='angstrom'),
        puw.get_value(expected, to_unit='angstrom'),
        rtol=0,
        atol=1e-12,
    )
    assert declarations['prepared_inputs'][key]['charge_unit'] == 'elementary_charge'
    np.testing.assert_allclose(prepared.charges, old['charges'], rtol=0, atol=1e-12)
assert list(msm.get(source, element='atom', atom_id=True)) == source_ids
np.testing.assert_allclose(
    puw.get_value(msm.get(source, coordinates=True), to_unit='angstrom'),
    source_coordinates,
    rtol=0,
    atol=1e-12,
)
print(
    {
        'original_prepared_values_match': True,
        'coordinate_tolerance_angstrom': 1e-12,
        'original_observed_source_preserved': True,
    }
)

## Declare boxes and admit the planned protocol

MolSysMT computes the reference centre. BindingMT owns the explicit 20 angstrom
boxes: native centre and the existing 60 angstrom +x displaced control. This is
an explicit-box choice, not a new molecular transformation or richer-domain
approximation. Planned box values are not claimed as an executed Vina projection.
The public protocol check admits the supported intent before any engine call.

In [ ]:
centre = msm.structure.get_center(reference, weights=None)[0, 0]
boxes = {
    'native': dmt.BoxRegion(
        center=centre, lengths=puw.quantity([20, 20, 20], 'angstrom')
    ),
    'displaced': dmt.BoxRegion(
        center=centre + puw.quantity([60, 0, 0], 'angstrom'),
        lengths=puw.quantity([20, 20, 20], 'angstrom'),
    ),
}
for name, box in boxes.items():
    assert all(
        box.contains(point) == (name == 'native')
        for point in msm.get(reference, coordinates=True)[0]
    )
protocol = dmt.VinaProtocol(
    seed=7,
    exhaustiveness=1,
    cpu=1,
    n_poses=9,
    scoring='vina',
    energy_range=puw.quantity(3, 'kcal/mol'),
    capture_backend_inputs=True,
    active_torsion_bonds=[],
)
problems = {
    name: dmt.DockingProblem(
        prepared_receptor,
        prepared_partner,
        box,
        metadata={
            'walkthrough': 'declared 3PTB rigid preparation',
            'source_file_sha256_declared_by_caller': source_sha256,
            'declaration_file_sha256': declaration_sha256,
            'molecular_hypothesis': 'dry HID57/HIE40/HIE91, CYX, charged termini, benzamidinium +1',
            'named_charges': charge_options,
            'named_types': typing_options,
        },
    )
    for name, box in boxes.items()
}
for problem in problems.values():
    protocol.validate_problem(problem)
    for role in ('receptor', 'partner'):
        assert problem.to_dict()['molecular_inputs'][role]['source_fingerprint'] is None
print(
    {
        'protocol_admitted': True,
        'planned_cases': list(problems),
        'executed_cases': [],
        'protocol_parameters': protocol.parameters,
    }
)

## Retain reports and declarations without inventing a molecular snapshot

All returned template/H reports, original observed attribute arrays and the
prepared BindingMT declarations are retained. Standard JSON IO below handles
NumPy containers; an actual quantity uses PyUnitWizard's sealed record codec
rather than stripping a unit. No native MolSys is serialized by this adapter.
Readers of such quantity records must use the provider's unit/field handshake.

Prepared JSON and PDBQT remain declarations/representations, not a complete native
system. Replaying the source recipe or supplying the actual native/prepared
objects remains necessary. Full molecular persistence belongs to MolSysMT.
Existing files are never overwritten.

In [ ]:
def record_json_default(value):
    if puw.is_quantity(value):
        return puw.convert(value, to_form='record').to_dict()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    raise TypeError(f'Unsupported report value: {type(value).__name__}')


provider_reports = {
    'schema': 'bindingmt.3ptb_preparation_reports@1',
    'source_sha256': source_sha256,
    'declaration_sha256': declaration_sha256,
    'versions': versions,
    'peptide_factory_report': factory['report'],
    'receptor_template_assessment': receptor_assessment,
    'ligand_template_assessment': ligand_assessment,
    'receptor_template_application': receptor_application['report'],
    'ligand_template_application': ligand_application['report'],
    'hydrogens': {role: value['report'] for role, value in hydrogen_outputs.items()},
    'original_observed_attributes': original_attributes,
    'charge_audits': charge_audits,
    'preparation_assessments': preparation_assessments,
    'receptor_generated_H_naming': {
        'rule': 'H plus three uppercase hexadecimal output-index digits',
        'indices': generated_h,
    },
}
artifacts = {}
for filename, value in [
    ('provider_reports.json', provider_reports),
    (
        'prepared_declarations.json',
        {role: prepared.to_dict() for role, prepared in preparations.items()},
    ),
    (
        'execution_plan.json',
        {
            'status': 'not_executed',
            'protocol': protocol.to_dict(),
            'problems': {name: problem.to_dict() for name, problem in problems.items()},
        },
    ),
]:
    payload = json.dumps(
        value, default=record_json_default, allow_nan=False, indent=1
    ).encode()
    with (output / filename).open('xb') as stream:
        stream.write(payload)
    artifacts[filename] = {
        'bytes': len(payload),
        'sha256': hashlib.sha256(payload).hexdigest(),
    }
for role, prepared in preparations.items():
    payload = prepared.to_pdbqt().encode()
    filename = role + '.pdbqt'
    with (output / filename).open('xb') as stream:
        stream.write(payload)
    artifacts[filename] = {
        'bytes': len(payload),
        'sha256': hashlib.sha256(payload).hexdigest(),
    }
assert hashlib.sha256(source_path.read_bytes()).hexdigest() == source_sha256
assert hashlib.sha256(declaration_path.read_bytes()).hexdigest() == declaration_sha256
receipt = {
    'schema': 'bindingmt.installed_3ptb_preparation@1',
    'scope': 'preparation_and_protocol_admission_only',
    'source_file_sha256': source_sha256,
    'declaration_file_sha256': declaration_sha256,
    'reader_versions': versions,
    'python': platform.python_version(),
    'source_counts': counts,
    'prepared_atoms': {'receptor': 2011, 'partner': 13},
    'projected_charge_unit': 'elementary_charge',
    'projected_charges': {
        role: audit['total_charge'] for role, audit in charge_audits.items()
    },
    'coordinate_tolerance_angstrom': 1e-12,
    'source_and_prepared_values_verified': True,
    'invalid_map_refusal': map_refusal,
    'artifacts': artifacts,
    'planned_cases': list(problems),
    'execution_status': 'not_executed',
    'molecular_snapshot_created': False,
    'chemical_model_suitability': 'unassessed',
    'C6_accepted': False,
}
with (output / 'receipt.json').open('x') as stream:
    json.dump(receipt, stream, indent=2, allow_nan=False)
print(json.dumps(receipt, indent=2, allow_nan=False))

## Execution follow-up — not executed by Run All

The native and displaced problems now exist in the live kernel. A follow-up must
be registered as a **new, separate two-search workflow demonstration** before
running, retaining actual inputs, outputs, producer identity, failures and counts.
The proposed scope is one rigid/native and one rigid/displaced call, seed 7,
exhaustiveness 1, CPU 1, at most nine returned poses each, with no retries,
rescoring, optimisation or new chemistry. This is not another execution of the
old 24-row benchmark and does not by itself establish affinity or recovery.

After that registration, the public calls would be:

```python
native_result = dmt.dock(problems['native'], protocol=protocol)
displaced_result = dmt.dock(problems['displaced'], protocol=protocol)
```

No executable cell above calls them. The [result-inspection notebook](installed_result_inspection.ipynb)
shows the subsequent saved-record audit, schema admission and unit/provenance
meaning. Reference evaluation needs its own explicit atom population and
correspondence: a nine-heavy-atom reference is not interchangeable with the
13-atom prepared ligand. Keep molecular selection/geometry in MolSysMT and
interpretation in BindingMT. Do not copy a private heavy-pose projection helper.

No viewer is opened here. Interactive observation remains deferred, as do public
package/platform claims and general MolecularMechanics persistence.